In [1]:
#/////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////
#                                              Import Packages                                                                       #
#/////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////

import boto3
import json
import pyodbc
import pandas as pd
from datetime import date, datetime, timedelta
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np
import plotly.express as px
import mplcursors
import plotly.graph_objects as go
from pandas.tseries.holiday import USFederalHolidayCalendar
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import TimeSeriesSplit

In [2]:
#/////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////
#                                                          Set TODAY                                                                 #
#/////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////

# For automation use:
# TODAY = pd.Timestamp(date.today())

# For backtesting, set manually:
TODAY = pd.Timestamp("2026-04-01")
TODAY = pd.to_datetime(TODAY)
print("Today's date is:", TODAY)

Today's date is: 2026-04-01 00:00:00


In [3]:
#/////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////
#                                     Connect to SQL Server: Load in Historical Data                                                 #
#/////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////

def get_secret(secret_name, region_name):
    """
    Retrieve secrets from AWS Secrets Manager.
    """
    try:
        session = boto3.session.Session()
        client  = session.client(service_name="secretsmanager", region_name=region_name)
        response = client.get_secret_value(SecretId=secret_name)
        secret   = json.loads(response["SecretString"])
        return secret
    except Exception as e:
        print(f"Failed to retrieve secret: {e}")
        return None

secret_name = "analytics/prod/credentials"
region_name = "us-east-1"

secret = get_secret(secret_name, region_name)

if secret:
    try:
        server   = secret["datamart"]["host"]
        port     = secret["datamart"]["port"]
        database = secret["datamart"]["db"]
        username = secret["datamart"]["user"]
        password = secret["datamart"]["password"]

        conn_str = (
            f"Driver={{ODBC Driver 17 for SQL Server}};"
            f"Server={server},{port};"
            f"Database={database};"
            f"UID={username};"
            f"PWD={password};"
        )

        select_query = """
SET DATEFIRST 1;  -- Monday = 1

WITH channels AS (
    SELECT DISTINCT
        CASE
            WHEN channel LIKE '%corr%' THEN 'Correspondent'
            ELSE channel
        END AS channel
    FROM marketing_sandbox.dbo.SDS WITH (NOLOCK)
    WHERE channel NOT LIKE '%Credit Union Partners%'
),

base AS (
    SELECT
        CASE
            WHEN s.channel LIKE '%corr%' THEN 'Correspondent'
            ELSE s.channel
        END AS channel,
        s.funded_date,
        s.loan_amount,
        DATEPART(YEAR, s.funded_date) AS year_of_funding,
        DATEPART(WEEKDAY, s.funded_date) - 1 AS day_of_week,
        DATEPART(DAY, s.funded_date) AS day_of_month,
        DATEDIFF(WEEK, DATEADD(MONTH, DATEDIFF(MONTH, 0, s.funded_date), 0), s.funded_date) + 1 AS week_of_month,
        DATEPART(MONTH, s.funded_date) AS month_of_year
    FROM marketing_sandbox.dbo.SDS s WITH (NOLOCK)
    WHERE s.funded_date >= DATEADD(YEAR, -5, CAST(GETDATE() AS DATE))
      AND s.funded_date <= (
            SELECT MIN(Calendar_Date)
            FROM (
                SELECT Calendar_Date,
                       ROW_NUMBER() OVER (ORDER BY Calendar_Date ASC) AS rn
                FROM marketing_sandbox.dbo.Calendar WITH (NOLOCK)
                WHERE Calendar_Date > CAST(GETDATE() AS DATE)
                  AND Biz_Day = 1
            ) biz
            WHERE rn = 3
      )
      AND s.funded_date IS NOT NULL
      AND s.channel NOT LIKE '%Credit Union Partners%'
),

freq AS (
    SELECT
        channel,
        funded_date,
        loan_amount,
        year_of_funding,
        day_of_week,
        day_of_month,
        week_of_month,
        month_of_year,
        COUNT(CASE WHEN day_of_week < 5 THEN 1 END)
            OVER (PARTITION BY channel, year_of_funding, month_of_year, week_of_month) AS loans_in_week,
        COUNT(CASE WHEN day_of_week < 5 THEN 1 END)
            OVER (PARTITION BY channel, year_of_funding, month_of_year) AS loans_in_month,
        COUNT(CASE WHEN day_of_week < 5 THEN 1 END)
            OVER (PARTITION BY channel, year_of_funding, month_of_year, week_of_month, day_of_week) AS loans_on_day_of_week,
        COUNT(CASE WHEN day_of_week < 5 THEN 1 END)
            OVER (PARTITION BY channel, year_of_funding, month_of_year, day_of_month) AS loans_on_day_of_month,
        SUM(CASE WHEN day_of_week < 5 THEN loan_amount ELSE 0 END)
            OVER (PARTITION BY channel, year_of_funding, month_of_year, week_of_month) AS amount_in_week,
        SUM(CASE WHEN day_of_week < 5 THEN loan_amount ELSE 0 END)
            OVER (PARTITION BY channel, year_of_funding, month_of_year) AS amount_in_month,
        SUM(CASE WHEN day_of_week < 5 THEN loan_amount ELSE 0 END)
            OVER (PARTITION BY channel, year_of_funding, month_of_year, week_of_month, day_of_week) AS amount_on_day_of_week,
        SUM(CASE WHEN day_of_week < 5 THEN loan_amount ELSE 0 END)
            OVER (PARTITION BY channel, year_of_funding, month_of_year, day_of_month) AS amount_on_day_of_month,
        SUM(CASE WHEN day_of_week < 5 THEN 1 ELSE 0 END)
            OVER (PARTITION BY channel, month_of_year) AS total_loans_in_month_across_years,
        SUM(CASE WHEN day_of_week < 5 THEN loan_amount ELSE 0 END)
            OVER (PARTITION BY channel, month_of_year) AS total_amount_in_month_across_years,
        SUM(CASE WHEN day_of_week < 5 THEN 1 ELSE 0 END)
            OVER (PARTITION BY channel, year_of_funding) AS total_loans_in_year,
        SUM(CASE WHEN day_of_week < 5 THEN loan_amount ELSE 0 END)
            OVER (PARTITION BY channel, year_of_funding) AS total_amount_in_year
    FROM base
),

agg_loans AS (
    SELECT
        channel,
        CAST(funded_date AS DATE) AS funded_date,
        year_of_funding,
        day_of_week,
        day_of_month,
        week_of_month,
        month_of_year,
        COUNT(*) AS number_of_loans,
        SUM(loan_amount) AS total_loan_amount,
        CAST(loans_in_week AS FLOAT) / NULLIF(loans_in_month, 0) AS week_weight,
        CAST(loans_on_day_of_week AS FLOAT) / NULLIF(loans_in_week, 0) AS day_of_week_weight,
        CAST(loans_on_day_of_month AS FLOAT) / NULLIF(loans_in_month, 0) AS day_of_month_weight,
        CAST(amount_in_week AS FLOAT) / NULLIF(amount_in_month, 0) AS week_amount_weight,
        CAST(amount_on_day_of_week AS FLOAT) / NULLIF(amount_in_week, 0) AS day_of_week_amount_weight,
        CAST(amount_on_day_of_month AS FLOAT) / NULLIF(amount_in_month, 0) AS day_of_month_amount_weight,
        CAST(total_loans_in_month_across_years AS FLOAT) / NULLIF(SUM(total_loans_in_month_across_years)
            OVER (PARTITION BY channel, month_of_year), 0) AS month_to_month_weight_loans,
        CAST(total_amount_in_month_across_years AS FLOAT) / NULLIF(SUM(total_amount_in_month_across_years)
            OVER (PARTITION BY channel, month_of_year), 0) AS month_to_month_weight_amount,
        CAST(total_loans_in_month_across_years AS FLOAT) / NULLIF(total_loans_in_year, 0) AS month_within_year_weight_loans,
        CAST(total_amount_in_month_across_years AS FLOAT) / NULLIF(total_amount_in_year, 0) AS month_within_year_weight_amount
    FROM freq
    GROUP BY
        channel, CAST(funded_date AS DATE), year_of_funding, day_of_week,
        day_of_month, week_of_month, month_of_year, loans_in_week, loans_in_month,
        loans_on_day_of_week, loans_on_day_of_month, amount_in_week, amount_in_month,
        amount_on_day_of_week, amount_on_day_of_month, total_loans_in_month_across_years,
        total_amount_in_month_across_years, total_loans_in_year, total_amount_in_year
),

calendar_channels AS (
    SELECT
        c.Calendar_Date, ch.channel, c.Biz_Day, c.Funding_Day,
        c.Funding_Day_Remaining_in_Month, c.Biz_Day_Remaining_in_Month,
        c.Is_Holiday,
        DATEPART(WEEKDAY, c.Calendar_Date) - 1 AS day_of_week,
        DATEPART(DAY, c.Calendar_Date) AS day_of_month,
        DATEDIFF(WEEK, DATEADD(MONTH, DATEDIFF(MONTH, 0, c.Calendar_Date), 0), c.Calendar_Date) + 1 AS week_of_month,
        DATEPART(MONTH, c.Calendar_Date) AS month_of_year
    FROM marketing_sandbox.dbo.Calendar c
    CROSS JOIN channels ch
    WHERE c.Calendar_Date >= DATEADD(YEAR, -5, CAST(GETDATE() AS DATE))
      AND c.Calendar_Date <= (
            SELECT MIN(Calendar_Date)
            FROM (
                SELECT Calendar_Date,
                       ROW_NUMBER() OVER (ORDER BY Calendar_Date ASC) AS rn
                FROM marketing_sandbox.dbo.Calendar WITH (NOLOCK)
                WHERE Calendar_Date > CAST(GETDATE() AS DATE)
                  AND Biz_Day = 1
            ) biz
            WHERE rn = 3
      )
),

joined AS (
    SELECT
        cc.Calendar_Date, cc.channel, cc.Biz_Day, cc.Funding_Day,
        cc.Funding_Day_Remaining_in_Month, cc.Biz_Day_Remaining_in_Month,
        cc.Is_Holiday, cc.day_of_week, cc.day_of_month, cc.week_of_month,
        cc.month_of_year,
        ISNULL(al.number_of_loans, 0)            AS number_of_loans,
        ISNULL(al.total_loan_amount, 0)           AS total_loan_amount,
        ISNULL(al.week_weight, 0)                 AS week_weight,
        ISNULL(al.day_of_week_weight, 0)          AS day_of_week_weight,
        ISNULL(al.day_of_month_weight, 0)         AS day_of_month_weight,
        ISNULL(al.week_amount_weight, 0)          AS week_amount_weight,
        ISNULL(al.day_of_week_amount_weight, 0)   AS day_of_week_amount_weight,
        ISNULL(al.day_of_month_amount_weight, 0)  AS day_of_month_amount_weight,
        ISNULL(al.month_to_month_weight_loans, 0) AS month_to_month_weight_loans,
        ISNULL(al.month_to_month_weight_amount, 0)AS month_to_month_weight_amount,
        ISNULL(al.month_within_year_weight_loans, 0)  AS month_within_year_weight_loans,
        ISNULL(al.month_within_year_weight_amount, 0) AS month_within_year_weight_amount
    FROM calendar_channels cc
    LEFT JOIN agg_loans al
        ON cc.Calendar_Date = al.funded_date
       AND cc.channel = al.channel
)

SELECT
    Calendar_Date, channel, Biz_Day, Funding_Day,
    Funding_Day_Remaining_in_Month, Biz_Day_Remaining_in_Month,
    Is_Holiday, day_of_week, day_of_month, week_of_month, month_of_year,
    CASE WHEN day_of_week IN (5,6) THEN 0 ELSE number_of_loans END          AS number_of_loans,
    CASE WHEN day_of_week IN (5,6) THEN 0 ELSE total_loan_amount END         AS total_loan_amount,
    CASE WHEN day_of_week IN (5,6) THEN 0 ELSE week_weight END               AS week_weight,
    CASE WHEN day_of_week IN (5,6) THEN 0 ELSE day_of_week_weight END        AS day_of_week_weight,
    CASE WHEN day_of_week IN (5,6) THEN 0 ELSE day_of_month_weight END       AS day_of_month_weight,
    CASE WHEN day_of_week IN (5,6) THEN 0 ELSE week_amount_weight END        AS week_amount_weight,
    CASE WHEN day_of_week IN (5,6) THEN 0 ELSE day_of_week_amount_weight END AS day_of_week_amount_weight,
    CASE WHEN day_of_week IN (5,6) THEN 0 ELSE day_of_month_amount_weight END AS day_of_month_amount_weight,
    CASE WHEN day_of_week IN (5,6) THEN 0 ELSE month_to_month_weight_loans END  AS month_to_month_weight_loans,
    CASE WHEN day_of_week IN (5,6) THEN 0 ELSE month_to_month_weight_amount END AS month_to_month_weight_amount,
    CASE WHEN day_of_week IN (5,6) THEN 0 ELSE month_within_year_weight_loans END  AS month_within_year_weight_loans,
    CASE WHEN day_of_week IN (5,6) THEN 0 ELSE month_within_year_weight_amount END AS month_within_year_weight_amount
FROM joined
ORDER BY Calendar_Date ASC, channel;
"""

        conn    = pyodbc.connect(conn_str)
        cursor  = conn.cursor()
        cursor.execute(select_query)
        columns = [column[0] for column in cursor.description]
        data    = cursor.fetchall()
        df      = pd.DataFrame.from_records(data, columns=columns)
        print(f"Data loaded successfully. Shape: {df.shape}")
        cursor.close()
        conn.close()

    except Exception as e:
        print(f"An error occurred: {e}")
        if 'conn' in locals():
            conn.close()
else:
    print("Failed to retrieve secrets. Check your configuration.")

Data loaded successfully. Shape: (9160, 23)


In [4]:
#/////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////
#                                     Connect to SQL Server: Load in Historical Data                                                 #
#/////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////

def get_secret(secret_name, region_name):
    """
    Retrieve secrets from AWS Secrets Manager.
    """
    try:
        session = boto3.session.Session()
        client  = session.client(service_name="secretsmanager", region_name=region_name)
        response = client.get_secret_value(SecretId=secret_name)
        secret   = json.loads(response["SecretString"])
        return secret
    except Exception as e:
        print(f"Failed to retrieve secret: {e}")
        return None

secret_name = "analytics/prod/credentials"
region_name = "us-east-1"

secret = get_secret(secret_name, region_name)

if secret:
    try:
        server   = secret["datamart"]["host"]
        port     = secret["datamart"]["port"]
        database = secret["datamart"]["db"]
        username = secret["datamart"]["user"]
        password = secret["datamart"]["password"]

        conn_str = (
            f"Driver={{ODBC Driver 17 for SQL Server}};"
            f"Server={server},{port};"
            f"Database={database};"
            f"UID={username};"
            f"PWD={password};"
        )

        select_query = """

WITH app_counts AS (
    SELECT 
        CAST(unified_app_date AS DATE) AS dt,
        channel,
        COUNT(*) AS application_count
    FROM marketing_sandbox.dbo.SDS WITH (NOLOCK)
    WHERE unified_app_date IS NOT NULL
    GROUP BY CAST(unified_app_date AS DATE), channel
),

uw_event_counts AS (
    SELECT 
        CAST(underwriting_submission_date AS DATE) AS dt,
        channel,
        COUNT(*) AS underwriting_submission_events
    FROM marketing_sandbox.dbo.SDS WITH (NOLOCK)
    WHERE underwriting_submission_date IS NOT NULL
    GROUP BY CAST(underwriting_submission_date AS DATE), channel
),

approval_event_counts AS (
    SELECT 
        CAST(initial_conditional_approval_date AS DATE) AS dt,
        channel,
        COUNT(*) AS approval_events
    FROM marketing_sandbox.dbo.SDS WITH (NOLOCK)
    WHERE initial_conditional_approval_date IS NOT NULL
    GROUP BY CAST(initial_conditional_approval_date AS DATE), channel
),

base AS (
    SELECT
        CAST(a.filedt AS DATE) AS filedt,
        a.channel,

        COUNT(DISTINCT a.loan_number) AS loan_count,
        SUM(a.loan_amount) AS loan_volume,

        COUNT(DISTINCT CASE WHEN b.funded_date IS NOT NULL THEN a.loan_number END) AS funded_loan_count,
        SUM(CASE WHEN b.funded_date IS NOT NULL THEN a.loan_amount ELSE 0 END) AS funded_loan_volume,

        -- status-based
        COUNT(DISTINCT CASE 
            WHEN b.underwriting_submission_date IS NOT NULL 
            THEN a.loan_number 
        END) AS underwriting_submission_count,

        COUNT(DISTINCT CASE 
            WHEN b.initial_conditional_approval_date IS NOT NULL 
            THEN a.loan_number 
        END) AS initial_conditional_approval_count,

        -- event-based
        ISNULL(uw.underwriting_submission_events, 0) AS underwriting_submission_events,
        ISNULL(ap.approval_events, 0) AS approval_events,
        ISNULL(ac.application_count, 0) AS application_count,

        -- pull-through
        CAST(
            100.0 * COUNT(DISTINCT CASE WHEN b.funded_date IS NOT NULL THEN a.loan_number END)
            / COUNT(DISTINCT a.loan_number)
            AS DECIMAL(10,2)
        ) AS pull_through_pct_count,

        CAST(
            100.0 * SUM(CASE WHEN b.funded_date IS NOT NULL THEN a.loan_amount ELSE 0 END)
            / NULLIF(SUM(a.loan_amount), 0)
            AS DECIMAL(10,2)
        ) AS pull_through_pct_volume

    FROM marketing_sandbox.dbo.skinny_core a WITH (NOLOCK)

    JOIN marketing_sandbox.dbo.SDS b WITH (NOLOCK)
        ON a.loan_number = b.loan_number

    LEFT JOIN app_counts ac
        ON CAST(a.filedt AS DATE) = ac.dt
        AND a.channel = ac.channel

    LEFT JOIN uw_event_counts uw
        ON CAST(a.filedt AS DATE) = uw.dt
        AND a.channel = uw.channel

    LEFT JOIN approval_event_counts ap
        ON CAST(a.filedt AS DATE) = ap.dt
        AND a.channel = ap.channel

    WHERE repeat_application = 0
      AND a.loan_status NOT LIKE '%C%'
      AND a.loan_status NOT IN (
            '0110','0115','0120','0125','0130','0140','0150','0160','0170','0180','0182',
            '0189','0190','0191','0192','0360','0362','0370','0375','0510','0610',
            '0618','0710','0711','0715','0720','1040'
      )

    GROUP BY 
        CAST(a.filedt AS DATE),
        a.channel,
        ac.application_count,
        uw.underwriting_submission_events,
        ap.approval_events
)

SELECT
    b.*,

    -- Calendar fields
    c.Biz_Day,
    c.Biz_Day_Remaining_in_Month,
    c.Biz_Days_in_Month,
    c.Is_Holiday,
    c.Is_Company_Holiday,
    c.Calendar_Days_Remaining

FROM base b

LEFT JOIN marketing_sandbox.dbo.Calendar c
    ON b.filedt = CAST(c.Calendar_Date AS DATE)

WHERE c.Is_Weekday = 1

ORDER BY 
    b.channel,
    b.filedt;
"""

        conn    = pyodbc.connect(conn_str)
        cursor  = conn.cursor()
        cursor.execute(select_query)
        columns = [column[0] for column in cursor.description]
        data    = cursor.fetchall()
        pipeline_df      = pd.DataFrame.from_records(data, columns=columns)
        print(f"Data loaded successfully. Shape: {pipeline_df.shape}")
        cursor.close()
        conn.close()

    except Exception as e:
        print(f"An error occurred: {e}")
        if 'conn' in locals():
            conn.close()
else:
    print("Failed to retrieve secrets. Check your configuration.")

Data loaded successfully. Shape: (1599, 19)


In [5]:
#/////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////
#                                                          Data Preprocessing                                                        #
#/////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////

df['Calendar_Date'] = pd.to_datetime(df['Calendar_Date'])
df.fillna(0, inplace=True)

pipeline_df['filedt'] = pd.to_datetime(pipeline_df['filedt'])
pipeline_df.fillna(0, inplace=True)

In [6]:
#/////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////
#                                                          Split by Channel                                                          #
#/////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////

wholesale_historical    = df[df['channel'] == 'Wholesale'].copy()
retail_historical       = df[df['channel'] == 'Retail'].copy()
retail_broker_historical= df[df['channel'] == 'Retail Broker'].copy()
corr_historical         = df[df['channel'] == 'Correspondent'].copy()

pipeline_wholesale    = pipeline_df[pipeline_df['channel'] == 'Wholesale'].copy()
pipeline_retail       = pipeline_df[pipeline_df['channel'] == 'Retail'].copy()
pipeline_retail_broker = pipeline_df[pipeline_df['channel'] == 'Retail Broker'].copy()

In [7]:
#////////////////////////////////////////////////////////////////////////////////////////////////////////
# Cast Decimal columns to float
#////////////////////////////////////////////////////////////////////////////////////////////////////////

# SQL DECIMAL/NUMERIC types arrive as Python decimal.Decimal — pandas can't do arithmetic on them
decimal_cols = pipeline_wholesale.select_dtypes(include=['object']).columns.tolist()

# Force all numeric-looking columns to float
for col in pipeline_wholesale.columns:
    if col not in ['filedt', 'channel', 'year_month']:
        pipeline_wholesale[col] = pd.to_numeric(pipeline_wholesale[col], errors='coerce').astype(float)

In [8]:
#////////////////////////////////////////////////////////////////////////////////////////////////////////
#                                   MASTER HOLIDAY CALENDAR (2021–2029)
#////////////////////////////////////////////////////////////////////////////////////////////////////////

def build_us_holiday_calendar(start_date='2021-01-01', end_date='2029-12-31'):
    """
    Builds full daily calendar spine with hard holiday flags (Is_Holiday).
    """
    full_dates = pd.date_range(start=start_date, end=end_date, freq='D')
    calendar   = pd.DataFrame({'Calendar_Date': full_dates})
    calendar['Calendar_Date'] = pd.to_datetime(calendar['Calendar_Date'])

    cal      = USFederalHolidayCalendar()
    holidays = cal.holidays(start=start_date, end=end_date)

    calendar['Is_Holiday'] = calendar['Calendar_Date'].isin(holidays).astype(int)
    return calendar[['Calendar_Date', 'Is_Holiday']]

In [9]:
#////////////////////////////////////////////////////////////////////////////////////////////////////////
#                          HolidayDistortionLearner
#  Learns per-(holiday_dow, business_day_offset) distortion factors from history.
#  e.g. "The Tuesday after a Monday holiday historically funds +35% above normal"
#////////////////////////////////////////////////////////////////////////////////////////////////////////

class HolidayDistortionLearner:
    """
    Learns empirical holiday distortion factors from historical data.

    For each holiday in training history:
      - Identifies the holiday's day-of-week (holiday_dow)
      - Looks at surrounding business days in a window [-2, +4]
      - Computes ratio of actual / model-expected for each offset day
      - Aggregates across years using median + shrinkage toward 1.0

    Result: distortion_table keyed by (holiday_dow, biz_offset)
    """

    def __init__(self,
                 window_before=2,
                 window_after=4,
                 shrinkage=0.3,
                 min_observations=2):

        self.window_before = window_before
        self.window_after = window_after
        self.shrinkage = shrinkage
        self.min_observations = min_observations
        self.distortion_table_ = {}

    def fit(self, df, holiday_calendar, expected_col='expected'):
        """
        df            : training dataframe with Calendar_Date, actual target, and expected column
        holiday_calendar : dataframe with Calendar_Date and Is_Holiday columns
        expected_col  : column name in df containing the model's baseline expected value
        """

        df = df.copy()
        df['Calendar_Date'] = pd.to_datetime(df['Calendar_Date'])

        holiday_calendar = holiday_calendar.copy()
        holiday_calendar['Calendar_Date'] = pd.to_datetime(
            holiday_calendar['Calendar_Date']
        )

        holiday_dates = holiday_calendar.loc[
            holiday_calendar['Is_Holiday'] == 1, 'Calendar_Date'
        ].tolist()

        # Business day spine from training data (weekdays only, non-holiday)
        biz_days = df.loc[
            (~df['Calendar_Date'].dt.dayofweek.isin([5, 6]))
        ].copy()

        biz_days = biz_days.sort_values('Calendar_Date').reset_index(drop=True)

        # Map date -> position in business day sequence
        biz_day_index = {
            row['Calendar_Date']: idx
            for idx, row in biz_days.iterrows()
        }

        # Accumulate ratios: {(holiday_dow, offset): [ratio, ratio, ...]}
        ratio_accumulator = {}

        for hdate in holiday_dates:

            holiday_dow = hdate.dayofweek

            # Skip weekends (shouldn't happen with USFederalHolidayCalendar observed)
            if holiday_dow in [5, 6]:
                continue

            # Find surrounding business days
            for offset in range(-self.window_before, self.window_after + 1):

                if offset == 0:
                    continue  # Holiday itself is always hard zero

                # Step through calendar days to find the Nth business day offset
                candidate = hdate
                steps = 0
                direction = 1 if offset > 0 else -1
                abs_offset = abs(offset)

                while steps < abs_offset:
                    candidate = candidate + pd.Timedelta(days=direction)
                    if candidate.dayofweek not in [5, 6]:
                        steps += 1

                # Look up actual and expected for this candidate date
                row = df.loc[df['Calendar_Date'] == candidate]

                if row.empty:
                    continue

                actual = row[expected_col.replace('expected', df.columns[
                    [c for c in df.columns if 'loan_amount' in c or 'loans' in c][0]
                    if any('loan_amount' in c or 'loans' in c for c in df.columns)
                    else -1
                ])].values[0] if False else None

                # Simpler: use whatever target column is available
                target_cols = [c for c in df.columns
                               if 'total_loan_amount' in c or 'number_of_loans' in c]

                if not target_cols:
                    continue

                target_col = target_cols[0]
                actual = row[target_col].values[0]
                expected = row[expected_col].values[0]

                if expected <= 0 or actual < 0:
                    continue

                ratio = actual / expected

                key = (holiday_dow, offset)

                if key not in ratio_accumulator:
                    ratio_accumulator[key] = []

                ratio_accumulator[key].append(ratio)

        # Aggregate: median + shrinkage toward 1.0
        self.distortion_table_ = {}

        for key, ratios in ratio_accumulator.items():

            if len(ratios) < self.min_observations:
                continue

            median_ratio = np.median(ratios)

            shrunk = (
                (1 - self.shrinkage) * median_ratio +
                self.shrinkage * 1.0
            )

            self.distortion_table_[key] = shrunk

        return self

    def get_distortion(self, holiday_dow, biz_offset):
        """
        Returns learned distortion factor for a given
        (holiday day-of-week, business day offset from holiday).
        Defaults to 1.0 if no data learned.
        """
        return self.distortion_table_.get((holiday_dow, biz_offset), 1.0)

In [10]:
#///////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////
#   Pipeline Data Preparation — compute monthly indicators + growth rates
#   for the regime detector
#///////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////

def prepare_pipeline_monthly(pipeline_channel_df):
    """
    Takes a daily pipeline dataframe for one channel and returns
    a monthly aggregation with the 3 indicators needed by the regime detector.

    Required input columns: filedt, application_count, approval_events,
    underwriting_submission_events, Biz_Days_in_Month, Is_Holiday, Is_Company_Holiday
    """
    pdf = pipeline_channel_df.copy()
    pdf['filedt'] = pd.to_datetime(pdf['filedt'])
    pdf['year_month'] = pdf['filedt'].dt.to_period('M')

    # Force numeric
    for col in pdf.columns:
        if col not in ['filedt', 'channel', 'year_month']:
            pdf[col] = pd.to_numeric(pdf[col], errors='coerce').astype(float)

    # ── Monthly sums ─────────────────────────────────────────────────────
    sum_cols = ['application_count', 'approval_events', 'underwriting_submission_events']
    monthly = pdf.groupby('year_month')[sum_cols].sum().reset_index()

    # ── Calendar aggregation ─────────────────────────────────────────────
    monthly_cal = pdf.groupby('year_month').agg(
        biz_days_in_month=('Biz_Days_in_Month', 'max'),
        holiday_count=('Is_Holiday', 'sum'),
        company_holiday_count=('Is_Company_Holiday', 'sum'),
    ).reset_index()

    monthly = monthly.merge(monthly_cal, on='year_month', how='left')
    monthly['effective_biz_days'] = (
        monthly['biz_days_in_month'] - monthly['holiday_count'] - monthly['company_holiday_count']
    )

    # ── Per-biz-day normalization ────────────────────────────────────────
    monthly['application_count_per_bizday'] = (
        monthly['application_count'] / monthly['effective_biz_days']
    )
    monthly['approval_events_per_bizday'] = (
        monthly['approval_events'] / monthly['effective_biz_days']
    )
    monthly['underwriting_submission_events_per_bizday'] = (
        monthly['underwriting_submission_events'] / monthly['effective_biz_days']
    )

    # ── 1-month growth rates ─────────────────────────────────────────────
    monthly['approval_events_per_bizday_growth_1m'] = (
        monthly['approval_events_per_bizday'].pct_change()
    )
    monthly['underwriting_submission_events_per_bizday_growth_1m'] = (
        monthly['underwriting_submission_events_per_bizday'].pct_change()
    )

    monthly['year_month_str'] = monthly['year_month'].astype(str)

    return monthly

In [11]:
#///////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////
#                          LeadingIndicatorRegimeDetector
#   Uses pipeline data (lead_1m) to classify the forecast month into a regime,
#   then returns the optimal Model A parameters for that regime.
#///////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////

class LeadingIndicatorRegimeDetector:
    """
    Classifies forecast months into regimes using a composite score
    built from 3 leading indicators (1-month lead):
      1. application_count_per_bizday
      2. approval_events_per_bizday_growth_1m
      3. underwriting_submission_events_per_bizday_growth_1m

    Regimes:
      January  → month == 1 (hard override — Dec holiday distortion corrupts recency)
      Regime1  → composite < threshold (steady/slow growth)
      Regime2  → composite >= threshold (accelerating growth)

    Backtest accuracy: 12/14 = 85.7% at threshold=0.30
    """

    # ── Regime parameter sets (from grid search backtest) ────────────────
    REGIME_PARAMS = {
        'January': dict(
            recency_strength=0.20,
            dow_shrinkage=0.25,
            interaction_shrinkage=0.35,
            seasonal_exponent=1.05,
            trend_dampening=0.70,
            forward_lift_daily=0.0025,
            level_lookback_days=60,
            trend_seasonal_tilt=0.04,
        ),
        'Regime1': dict(
            recency_strength=1.0,
            dow_shrinkage=0.10,
            interaction_shrinkage=0.20,
            seasonal_exponent=1.05,
            trend_dampening=0.70,
            forward_lift_daily=0.005,
            level_lookback_days=90,
            trend_seasonal_tilt=0.15,
        ),
        'Regime2': dict(
            recency_strength=0.68,
            dow_shrinkage=0.15,
            interaction_shrinkage=0.28,
            seasonal_exponent=1.02,
            trend_dampening=0.83,
            forward_lift_daily=0.0,
            level_lookback_days=60,
            trend_seasonal_tilt=0.04,
        ),
    }

    # ── Standardization anchors (fitted from Jan 2025 – Mar 2026 backtest) ──
    # These are the mean/std of each indicator across the 14 backtest months
    INDICATOR_STATS = {
        'application_count_per_bizday':                          {'mean': 175.93, 'std': 14.89},
        'approval_events_per_bizday_growth_1m':                  {'mean': -0.016, 'std': 0.107},
        'underwriting_submission_events_per_bizday_growth_1m':   {'mean': -0.018, 'std': 0.099},
    }

    def __init__(self, threshold=0.30):
        self.threshold = threshold

    def compute_composite(self, apps_per_bizday, appr_growth_1m, uw_growth_1m):
        """
        Compute standardized composite score from 3 indicators.
        Each is z-scored using backtest-period stats, then averaged.
        """
        z_apps = (apps_per_bizday - self.INDICATOR_STATS['application_count_per_bizday']['mean']) / \
                  self.INDICATOR_STATS['application_count_per_bizday']['std']
        z_appr = (appr_growth_1m - self.INDICATOR_STATS['approval_events_per_bizday_growth_1m']['mean']) / \
                  self.INDICATOR_STATS['approval_events_per_bizday_growth_1m']['std']
        z_uw   = (uw_growth_1m - self.INDICATOR_STATS['underwriting_submission_events_per_bizday_growth_1m']['mean']) / \
                  self.INDICATOR_STATS['underwriting_submission_events_per_bizday_growth_1m']['std']
        return (z_apps + z_appr + z_uw) / 3.0

    def detect_regime(self, forecast_month, pipeline_monthly_df):
        """
        Detect the regime for a given forecast month.

        Parameters
        ----------
        forecast_month : int
            Month number (1-12) of the forecast period.
        pipeline_monthly_df : pd.DataFrame
            Monthly aggregated pipeline data with columns:
              - year_month (Period[M])
              - application_count_per_bizday
              - approval_events_per_bizday_growth_1m
              - underwriting_submission_events_per_bizday_growth_1m
            The LAST ROW should be the most recent complete month
            (i.e., the 1-month-lagged indicator month).

        Returns
        -------
        regime : str
            'January', 'Regime1', or 'Regime2'
        composite : float
            The composite score used for classification
        params : dict
            Model A parameter dictionary for this regime
        """
        # ── January hard override ────────────────────────────────────────
        if forecast_month == 1:
            return 'January', float('nan'), self.REGIME_PARAMS['January'].copy()

        # ── Get the most recent complete month's indicators ──────────────
        latest = pipeline_monthly_df.iloc[-1]

        apps_per_biz = latest['application_count_per_bizday']
        appr_growth  = latest.get('approval_events_per_bizday_growth_1m', 0.0)
        uw_growth    = latest.get('underwriting_submission_events_per_bizday_growth_1m', 0.0)

        composite = self.compute_composite(apps_per_biz, appr_growth, uw_growth)

        if composite >= self.threshold:
            regime = 'Regime2'
        else:
            regime = 'Regime1'

        print(
            f"  [Regime] month={forecast_month}  composite={composite:+.3f}  "
            f"→ {regime}  (apps/biz={apps_per_biz:.1f}, "
            f"appr_g={appr_growth:+.3f}, uw_g={uw_growth:+.3f})"
        )

        return regime, composite, self.REGIME_PARAMS[regime].copy()

In [12]:
#///////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////
#                                         Model A — StructuralLoanForecaster_A                                                     #
#///////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////

class StructuralLoanForecaster_A:

    def __init__(self,
                 target='total_loan_amount',
                 dow_shrinkage=0.25,
                 interaction_shrinkage=0.35,
                 seasonal_exponent=1.02,
                 recency_strength=0.35,
                 trend_seasonal_tilt=0.08,
                 level_lookback_days=60,
                 trend_dampening=0.83,
                 forward_lift_daily=0.0025,
                 holiday_window_before=2,
                 holiday_window_after=4,
                 holiday_shrinkage=0.3):

        self.target = target
        self.dow_shrinkage = dow_shrinkage
        self.interaction_shrinkage = interaction_shrinkage
        self.seasonal_exponent = seasonal_exponent
        self.recency_strength = recency_strength
        self.trend_seasonal_tilt = trend_seasonal_tilt
        self.level_lookback_days = level_lookback_days
        self.trend_dampening = trend_dampening
        self.forward_lift_daily = forward_lift_daily
        self.holiday_window_before = holiday_window_before
        self.holiday_window_after = holiday_window_after
        self.holiday_shrinkage = holiday_shrinkage

        self.master_calendar = None
        self.distortion_table_ = {}
        self.fitted = False

    # ---------------------------------------------------
    # INTERNAL: compute expected baseline for a dataframe
    # ---------------------------------------------------
    def _compute_expected(self, df, t_array):
        """
        Given a df with dow/dom/moy columns and a t_array of
        trend indices, returns the baseline expected value array
        (trend × DOW × DOM × MOY × interaction).
        """

        trend = np.exp(
            self.trend_intercept + self.trend_slope * t_array
        )

        dow_e = df['dow'].map(self.dow_effect).fillna(1.0).values
        dom_e = df['dom'].map(self.dom_effect).fillna(1.0).values
        moy_e = df['moy'].map(self.moy_effect).fillna(1.0).values

        interaction_e = np.array([
            self.interaction_effect.get((d, m), 1.0)
            for d, m in zip(df['dow'], df['moy'])
        ])

        seasonal = dow_e * dom_e * moy_e * interaction_e
        seasonal = seasonal / np.mean(seasonal)
        seasonal = seasonal ** self.seasonal_exponent

        tilt = 1 + self.trend_seasonal_tilt * (seasonal - 1)

        return self.base_level * seasonal * trend * tilt

    # ---------------------------------------------------
    # FIT
    # ---------------------------------------------------
    def fit(self, df, holiday_calendar=None):

        df = df.copy()
        df = df.sort_values('Calendar_Date')
        df['Calendar_Date'] = pd.to_datetime(df['Calendar_Date'])

        df['dow'] = df['Calendar_Date'].dt.dayofweek
        df['dom'] = df['Calendar_Date'].dt.day
        df['moy'] = df['Calendar_Date'].dt.month
        df['is_weekend'] = df['dow'].isin([5, 6])

        # ---- Store master holiday calendar ----
        if holiday_calendar is not None:
            calendar = holiday_calendar.copy()
            calendar['Calendar_Date'] = pd.to_datetime(calendar['Calendar_Date'])
            self.master_calendar = calendar

            df = df.merge(
                calendar[['Calendar_Date', 'Is_Holiday']],
                on='Calendar_Date',
                how='left'
            )
            df['Is_Holiday'] = df['Is_Holiday'].fillna(0)

        else:
            df['Is_Holiday'] = 0
            self.master_calendar = None

        # ---- Production mask: weekdays, non-holiday, non-zero ----
        prod_mask = (
            (~df['is_weekend']) &
            (df['Is_Holiday'] == 0) &
            (df[self.target] > 0)
        )

        # ---- NEW: Seasonal mask — full completed months only ----
        last_date           = df['Calendar_Date'].max()
        partial_month_start = last_date.replace(day=1)

        seasonal_mask = (
            prod_mask &
            (df['Calendar_Date'] < partial_month_start)
        )

        df_prod = df.loc[prod_mask].copy()

        self.last_train_date = df['Calendar_Date'].max()
        self.n_train = len(df)

        # ---- Trend (log-linear WLS, recency weighted) — FAST VERSION ----
        t_prod = np.arange(len(df_prod))
        recency_prod = np.exp(self.recency_strength * (t_prod / len(df_prod)))

        y = np.log(df_prod[self.target].values)
        X = np.vstack([np.ones(len(t_prod)), t_prod]).T

        sqrt_w = np.sqrt(recency_prod)
        Xw     = X * sqrt_w[:, None]
        yw     = y * sqrt_w
        beta   = np.linalg.lstsq(Xw, yw, rcond=None)[0]

        self.trend_intercept = beta[0]
        self.trend_slope = beta[1]

        t_full = np.arange(len(df))
        trend_fitted = np.exp(self.trend_intercept + self.trend_slope * t_full)

        df['trend_fitted'] = trend_fitted
        df['detrended'] = df[self.target] / trend_fitted

        # ---- Base level — uses prod_mask (includes partial month) ----
        recent_prod = df.loc[prod_mask].tail(self.level_lookback_days)
        self.base_level = (
            recent_prod[self.target].mean() /
            recent_prod['trend_fitted'].mean()
        )

        # ---- DOW effect — uses seasonal_mask (full months only) ----
        dow_raw = df.loc[seasonal_mask].groupby('dow')['detrended'].mean()
        dow_shrunk = (
            (1 - self.dow_shrinkage) * dow_raw +
            self.dow_shrinkage * self.base_level
        )
        self.dow_effect = dow_shrunk / dow_shrunk.mean()
        self.dow_effect.loc[5] = 0.0
        self.dow_effect.loc[6] = 0.0

        df['after_dow'] = df['detrended'] / df['dow'].map(
            lambda x: self.dow_effect.get(x, 1.0)
        )

        # ---- DOM effect — uses seasonal_mask ----
        dom_raw = df.loc[seasonal_mask].groupby('dom')['after_dow'].mean()
        self.dom_effect = dom_raw / dom_raw.mean()
        df['after_dom'] = df['after_dow'] / df['dom'].map(self.dom_effect)

        # ---- MOY effect — uses seasonal_mask ----
        moy_raw = df.loc[seasonal_mask].groupby('moy')['after_dom'].mean()
        self.moy_effect = moy_raw / moy_raw.mean()
        df['after_moy'] = df['after_dom'] / df['moy'].map(self.moy_effect)

        # ---- DOW x MOY interaction — uses seasonal_mask ----
        interaction_raw = (
            df.loc[seasonal_mask]
            .groupby(['dow', 'moy'])['after_moy']
            .mean()
        )
        interaction_shrunk = (
            (1 - self.interaction_shrinkage) * interaction_raw +
            self.interaction_shrinkage * interaction_raw.mean()
        )
        self.interaction_effect = (
            interaction_shrunk / interaction_shrunk.mean()
        )

        # ================================================
        # LEARN HOLIDAY DISTORTION FACTORS — uses prod_mask
        # ================================================
        if self.master_calendar is not None:

            t_full_arr = np.arange(len(df))
            df['expected'] = self._compute_expected(df, t_full_arr)

            holiday_dates = self.master_calendar.loc[
                self.master_calendar['Is_Holiday'] == 1, 'Calendar_Date'
            ].tolist()

            biz_day_list = df.loc[
                ~df['is_weekend']
            ]['Calendar_Date'].sort_values().tolist()

            biz_day_pos = {d: i for i, d in enumerate(biz_day_list)}

            ratio_accumulator = {}

            for hdate in holiday_dates:

                holiday_dow = hdate.dayofweek

                if holiday_dow in [5, 6]:
                    continue

                if hdate not in biz_day_pos:
                    continue

                h_pos = biz_day_pos[hdate]

                for offset in range(
                    -self.holiday_window_before,
                    self.holiday_window_after + 1
                ):

                    if offset == 0:
                        continue

                    target_pos = h_pos + offset

                    if target_pos < 0 or target_pos >= len(biz_day_list):
                        continue

                    target_date = biz_day_list[target_pos]

                    row = df.loc[df['Calendar_Date'] == target_date]

                    if row.empty:
                        continue

                    if row['Is_Holiday'].values[0] == 1:
                        continue
                    if row['is_weekend'].values[0]:
                        continue

                    actual = row[self.target].values[0]
                    expected = row['expected'].values[0]

                    if expected <= 0 or actual <= 0:
                        continue

                    ratio = actual / expected
                    key = (holiday_dow, offset)

                    if key not in ratio_accumulator:
                        ratio_accumulator[key] = []

                    ratio_accumulator[key].append(ratio)

            self.distortion_table_ = {}

            for key, ratios in ratio_accumulator.items():

                if len(ratios) < 2:
                    continue

                median_ratio = float(np.median(ratios))

                shrunk = (
                    (1 - self.holiday_shrinkage) * median_ratio +
                    self.holiday_shrinkage * 1.0
                )

                self.distortion_table_[key] = shrunk

        self.fitted = True
        return self

    # ---------------------------------------------------
    # FORECAST
    # ---------------------------------------------------
    def forecast_from_date(self, start_date, months_forward=1):

        if not self.fitted:
            raise Exception('Model must be fit first.')

        start_date = pd.to_datetime(start_date)
        end_date = start_date + pd.DateOffset(months=months_forward)

        future_dates = pd.date_range(
            start=start_date,
            end=end_date - pd.Timedelta(days=1),
            freq='D'
        )

        df = pd.DataFrame({'Calendar_Date': future_dates})
        df['dow'] = df['Calendar_Date'].dt.dayofweek
        df['dom'] = df['Calendar_Date'].dt.day
        df['moy'] = df['Calendar_Date'].dt.month
        df['is_weekend'] = df['dow'].isin([5, 6])

        # ---- Trend ----
        t_future = np.arange(self.n_train, self.n_train + len(df))

        trend_multiplier = np.exp(
            self.trend_intercept +
            (self.trend_slope * self.trend_dampening) * t_future
        )

        horizon_index = np.arange(len(df))
        forward_lift = 1 + self.forward_lift_daily * horizon_index

        # ---- Seasonal ----
        dow_e = df['dow'].map(self.dow_effect).fillna(1.0).values
        dom_e = df['dom'].map(self.dom_effect).fillna(1.0).values
        moy_e = df['moy'].map(self.moy_effect).fillna(1.0).values

        interaction_e = np.array([
            self.interaction_effect.get((d, m), 1.0)
            for d, m in zip(df['dow'], df['moy'])
        ])

        seasonal = dow_e * dom_e * moy_e * interaction_e
        seasonal = seasonal / np.mean(seasonal)
        seasonal = seasonal ** self.seasonal_exponent

        tilt = 1 + self.trend_seasonal_tilt * (seasonal - 1)

        forecast = (
            self.base_level
            * seasonal
            * trend_multiplier
            * tilt
            * forward_lift
        )

        # ---- Weekend hard zero ----
        forecast[df['is_weekend'].values] = 0.0

        # ---- Apply holiday calendar ----
        if self.master_calendar is not None:

            df = df.merge(
                self.master_calendar,
                on='Calendar_Date',
                how='left'
            )
            df['Is_Holiday'] = df['Is_Holiday'].fillna(0)

            # Hard zero on holiday itself
            forecast[df['Is_Holiday'].values == 1] = 0.0

            holiday_dates_forecast = self.master_calendar.loc[
                (self.master_calendar['Is_Holiday'] == 1) &
                (self.master_calendar['Calendar_Date'] >= start_date) &
                (self.master_calendar['Calendar_Date'] <= future_dates[-1])
            ]['Calendar_Date'].tolist()

            lookback_start = start_date - pd.Timedelta(
                days=(self.holiday_window_before + 1) * 2
            )

            holiday_dates_nearby = self.master_calendar.loc[
                (self.master_calendar['Is_Holiday'] == 1) &
                (self.master_calendar['Calendar_Date'] >= lookback_start) &
                (self.master_calendar['Calendar_Date'] < start_date)
            ]['Calendar_Date'].tolist()

            all_relevant_holidays = holiday_dates_nearby + holiday_dates_forecast

            forecast_biz_days = df.loc[
                ~df['is_weekend']
            ]['Calendar_Date'].sort_values().tolist()

            forecast_biz_pos = {d: i for i, d in enumerate(forecast_biz_days)}

            distortion = np.ones(len(df))

            for hdate in all_relevant_holidays:

                holiday_dow = hdate.dayofweek

                if holiday_dow in [5, 6]:
                    continue

                for offset in range(
                    -self.holiday_window_before,
                    self.holiday_window_after + 1
                ):

                    if offset == 0:
                        continue

                    candidate = hdate
                    steps = 0
                    direction = 1 if offset > 0 else -1
                    abs_offset = abs(offset)

                    while steps < abs_offset:
                        candidate = candidate + pd.Timedelta(days=direction)
                        if candidate.dayofweek not in [5, 6]:
                            steps += 1

                    row_idx = df.index[df['Calendar_Date'] == candidate].tolist()

                    if not row_idx:
                        continue

                    idx = row_idx[0]

                    factor = self.distortion_table_.get(
                        (holiday_dow, offset), 1.0
                    )

                    distortion[idx] = factor

            forecast = forecast * distortion

        df['forecast'] = forecast

        return df

In [13]:
#///////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////
#                                         Model C — StructuralLoanForecaster_C (December Specialist)                              #
#///////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////

class StructuralLoanForecaster_C:
    """
    December-specialist structural forecaster for daily loan volumes.

    December is uniquely difficult because:
      - Volume collapses sharply after ~Dec 18 as borrowers/staff go offline
      - Early December often sees a catch-up surge from November pipeline
      - The Christmas–New Year holiday cluster is tightly spaced, causing
        cascading distortion across nearly every business day in the final 2 weeks

    Design decisions vs Model A/B:
      - Seasonal effects (DOW, DOM) fitted on December data ONLY,
        using all available historical Decembers (full months only)
      - Separate pre/post cliff base levels to capture the Dec 18 volume cliff
      - Holiday distortion window widened (before=2, after=6) to capture
        the full Christmas–NYE cascade — Dec 24 and Dec 31 suppression
        is learned empirically from history via distortion factors rather
        than being hard-coded
      - distortion_max_factor widened to 1.4 — December is more volatile
      - distortion_min_obs lowered to 2 — fewer historical Decembers
      - Trend fitted on all production data (not December-only) so the
        long-run growth rate is informed by the full history
    """

    def __init__(
        self,
        target                   = 'total_loan_amount',
        recency_strength         = 3.0,
        dow_shrinkage            = 0.15,
        interaction_shrinkage    = 0.25,
        level_lookback_days      = 60,
        holiday_window_before    = 2,
        holiday_window_after     = 6,
        holiday_shrinkage        = 0.2,
        distortion_min_obs       = 2,
        distortion_max_factor    = 1.4,
        trend_dampening          = 0.80,
        forward_lift_daily       = 0.0,
        cliff_date               = 18,
        cliff_shrinkage          = 0.5,
    ):
        self.target                = target
        self.recency_strength      = recency_strength
        self.dow_shrinkage         = dow_shrinkage
        self.interaction_shrinkage = interaction_shrinkage
        self.level_lookback_days   = level_lookback_days
        self.holiday_window_before = holiday_window_before
        self.holiday_window_after  = holiday_window_after
        self.holiday_shrinkage     = holiday_shrinkage
        self.distortion_min_obs    = distortion_min_obs
        self.distortion_max_factor = distortion_max_factor
        self.trend_dampening       = trend_dampening
        self.forward_lift_daily    = forward_lift_daily
        self.cliff_date            = cliff_date
        self.cliff_shrinkage       = cliff_shrinkage

        self.master_calendar   = None
        self.distortion_table_ = {}
        self.fitted            = False

    # ------------------------------------------------------------------
    def _compute_expected(self, df, t_arr):
        trend = np.exp(self.trend_intercept + self.trend_slope * t_arr)
        dow_e = df['dow'].map(self.dow_effect).fillna(1.0).values
        dom_e = df['dom'].map(self.dom_effect).fillna(1.0).values
        interaction_e = np.array([
            self.interaction_effect.get((d, 12), 1.0)
            for d in df['dow']
        ])
        return self.base_level * trend * dow_e * dom_e * interaction_e

    # ------------------------------------------------------------------
    def fit(self, df, holiday_calendar=None):

        df = df.copy()
        df = df.sort_values('Calendar_Date').reset_index(drop=True)
        df['Calendar_Date'] = pd.to_datetime(df['Calendar_Date'])
        df['dow']           = df['Calendar_Date'].dt.dayofweek
        df['dom']           = df['Calendar_Date'].dt.day
        df['moy']           = df['Calendar_Date'].dt.month
        df['is_weekend']    = df['dow'].isin([5, 6])

        # ---- Holiday calendar ----
        if holiday_calendar is not None:
            calendar = holiday_calendar.copy()
            calendar['Calendar_Date'] = pd.to_datetime(calendar['Calendar_Date'])
            self.master_calendar = calendar
            df = df.merge(
                calendar[['Calendar_Date', 'Is_Holiday']],
                on='Calendar_Date', how='left'
            )
            df['Is_Holiday'] = df['Is_Holiday'].fillna(0)
        else:
            df['Is_Holiday'] = 0
            self.master_calendar = None

        df = df.reset_index(drop=True)

        # ---- Production mask (all data — used for trend) ----
        prod_mask = (
            (~df['is_weekend']) &
            (df['Is_Holiday'] == 0) &
            (df[self.target] > 0)
        )

        # ---- Holiday halo — exclude halo days from seasonal fitting ----
        if self.master_calendar is not None:
            holiday_dates = set(pd.to_datetime(
                self.master_calendar.loc[
                    self.master_calendar['Is_Holiday'] == 1, 'Calendar_Date'
                ]
            ))
            halo_dates = set()
            for hdate in holiday_dates:
                for offset in range(
                    -self.holiday_window_before,
                     self.holiday_window_after + 1
                ):
                    candidate = hdate + pd.Timedelta(days=offset)
                    if candidate.dayofweek not in [5, 6]:
                        halo_dates.add(candidate)
            df['in_halo'] = df['Calendar_Date'].isin(halo_dates)
        else:
            holiday_dates = set()
            df['in_halo']  = False

        # ---- December-only seasonal mask ----
        # Full completed Decembers only, halo days excluded so the
        # Christmas/NYE distortion doesn't corrupt DOW/DOM weights
        last_date           = df['Calendar_Date'].max()
        partial_month_start = last_date.replace(day=1)

        dec_seasonal_mask = (
            prod_mask &
            (df['moy'] == 12) &
            (~df['in_halo']) &
            (df['Calendar_Date'] < partial_month_start)
        )

        self.last_train_date = df['Calendar_Date'].max()
        self.n_train         = len(df)

        # ---- Trend — fitted on ALL production data ----
        df_prod      = df.loc[prod_mask].copy()
        t_prod       = np.arange(len(df_prod))
        recency_prod = np.exp(self.recency_strength * (t_prod / len(df_prod)))
        y_log        = np.log(df_prod[self.target].values)
        X            = np.vstack([np.ones(len(t_prod)), t_prod]).T
        W            = np.diag(recency_prod)
        beta         = np.linalg.inv(X.T @ W @ X) @ (X.T @ W @ y_log)
        self.trend_intercept = beta[0]
        self.trend_slope     = beta[1]

        t_full       = np.arange(len(df))
        trend_fitted = np.exp(self.trend_intercept + self.trend_slope * t_full)
        df['trend_fitted'] = trend_fitted
        df['detrended']    = np.where(
            trend_fitted > 0, df[self.target] / trend_fitted, 0.0
        )

        # ---- Base level ----
        recent_prod     = df.loc[prod_mask].tail(self.level_lookback_days)
        target_mean     = float(recent_prod[self.target].mean())
        trend_mean      = float(recent_prod['trend_fitted'].mean())
        self.base_level = target_mean / trend_mean if trend_mean > 0 else 1.0

        # ---- Pre/post cliff base levels ----
        # Measures how much volume drops after cliff_date in December
        # so the forecast can scale down the back half of the month
        dec_pre_mask  = dec_seasonal_mask & (df['dom'] <= self.cliff_date)
        dec_post_mask = dec_seasonal_mask & (df['dom'] >  self.cliff_date)

        pre_mean  = df.loc[dec_pre_mask,  self.target].mean()
        post_mean = df.loc[dec_post_mask, self.target].mean()

        if pd.notna(pre_mean) and pd.notna(post_mean) and pre_mean > 0:
            self.cliff_ratio = post_mean / pre_mean
        else:
            self.cliff_ratio = 1.0

        print(f"  [Model C] cliff_ratio (post/pre Dec {self.cliff_date}): "
              f"{self.cliff_ratio:.3f}")

        # ---- DOW effect — December data only, halo excluded ----
        dow_raw    = df.loc[dec_seasonal_mask].groupby('dow')['detrended'].mean()
        dow_shrunk = (
            (1 - self.dow_shrinkage) * dow_raw +
            self.dow_shrinkage * self.base_level
        )
        self.dow_effect        = dow_shrunk / dow_shrunk.mean()
        self.dow_effect.loc[5] = 0.0
        self.dow_effect.loc[6] = 0.0

        df['after_dow'] = df['detrended'] / df['dow'].map(
            lambda x: self.dow_effect.get(x, 1.0)
        )

        # ---- DOM effect — December data only, halo excluded ----
        dom_raw         = df.loc[dec_seasonal_mask].groupby('dom')['after_dow'].mean()
        self.dom_effect = dom_raw / dom_raw.mean()
        df['after_dom'] = df['after_dow'] / df['dom'].map(self.dom_effect)

        # ---- DOW x December interaction ----
        interaction_raw    = (
            df.loc[dec_seasonal_mask]
            .groupby(['dow', 'moy'])['after_dom']
            .mean()
        )
        interaction_shrunk = (
            (1 - self.interaction_shrinkage) * interaction_raw +
            self.interaction_shrinkage * interaction_raw.mean()
        )
        self.interaction_effect = interaction_shrunk / interaction_shrunk.mean()

        # ---- Holiday distortion ----
        # wider window (after=6) captures the full Christmas–NYE cascade.
        # Dec 24 / Dec 31 suppression is learned here from history as
        # distortion factors rather than being hard-coded.
        if self.master_calendar is not None:

            # Use only December holidays as anchors
            dec_holidays = [h for h in holiday_dates if h.month == 12]

            t_full_arr     = np.arange(len(df))
            df['expected'] = self._compute_expected(df, t_full_arr)

            biz_day_list = df.loc[
                ~df['is_weekend']
            ]['Calendar_Date'].sort_values().tolist()
            biz_day_pos  = {d: i for i, d in enumerate(biz_day_list)}

            ratio_accumulator = {}

            for hdate in dec_holidays:
                holiday_dow = hdate.dayofweek
                if holiday_dow in [5, 6]:
                    continue
                if hdate not in biz_day_pos:
                    continue
                h_pos = biz_day_pos[hdate]

                for offset in range(
                    -self.holiday_window_before,
                     self.holiday_window_after + 1
                ):
                    if offset == 0:
                        continue
                    tp = h_pos + offset
                    if tp < 0 or tp >= len(biz_day_list):
                        continue
                    target_date = biz_day_list[tp]
                    row         = df.loc[df['Calendar_Date'] == target_date]
                    if row.empty:
                        continue
                    if row['Is_Holiday'].values[0] == 1:
                        continue
                    if row['is_weekend'].values[0]:
                        continue
                    actual   = row[self.target].values[0]
                    expected = row['expected'].values[0]
                    if expected <= 0 or actual <= 0:
                        continue
                    ratio = actual / expected
                    ratio_accumulator.setdefault(
                        (holiday_dow, offset), []
                    ).append(ratio)

            self.distortion_table_ = {}
            for key, ratios in ratio_accumulator.items():
                if len(ratios) < self.distortion_min_obs:
                    continue
                raw_median = float(np.median(ratios))
                shrunk     = (
                    (1 - self.holiday_shrinkage) * raw_median +
                    self.holiday_shrinkage * 1.0
                )
                capped = float(np.clip(
                    shrunk,
                    1.0 / self.distortion_max_factor,
                    self.distortion_max_factor
                ))
                self.distortion_table_[key] = capped

            print(f"  [Model C] Distortion keys learned: "
                  f"{sorted(self.distortion_table_.keys())}")
        else:
            self.distortion_table_ = {}

        self.fitted = True
        return self

    # ------------------------------------------------------------------
    def forecast_from_date(self, start_date, months_forward=1):

        if not self.fitted:
            raise Exception('Model must be fit before forecasting.')

        start_date   = pd.to_datetime(start_date)
        end_date     = start_date + pd.DateOffset(months=months_forward)
        future_dates = pd.date_range(
            start=start_date,
            end=end_date - pd.Timedelta(days=1),
            freq='D'
        )

        df = pd.DataFrame({'Calendar_Date': future_dates})
        df['dow']        = df['Calendar_Date'].dt.dayofweek
        df['dom']        = df['Calendar_Date'].dt.day
        df['moy']        = df['Calendar_Date'].dt.month
        df['is_weekend'] = df['dow'].isin([5, 6])

        # ---- Trend ----
        t_future         = np.arange(self.n_train, self.n_train + len(df))
        trend_multiplier = np.exp(
            self.trend_intercept +
            (self.trend_slope * self.trend_dampening) * t_future
        )

        horizon_index = np.arange(len(df))
        forward_lift  = 1 + self.forward_lift_daily * horizon_index

        # ---- Seasonal ----
        dow_e = df['dow'].map(self.dow_effect).fillna(1.0).values
        dom_e = df['dom'].map(self.dom_effect).fillna(1.0).values
        interaction_e = np.array([
            self.interaction_effect.get((d, 12), 1.0)
            for d in df['dow']
        ])

        seasonal = dow_e * dom_e * interaction_e
        seasonal = seasonal / np.mean(seasonal)

        forecast = (
            self.base_level * seasonal * trend_multiplier * forward_lift
        )

        # ---- Pre/post cliff scaling ----
        post_cliff_mask = (
            (df['moy'] == 12) & (df['dom'] > self.cliff_date)
        ).values
        forecast[post_cliff_mask] = (
            forecast[post_cliff_mask] *
            (self.cliff_ratio + (1 - self.cliff_ratio) * self.cliff_shrinkage)
        )

        # ---- Weekend hard zero ----
        forecast[df['is_weekend'].values] = 0.0

        # ---- Holiday calendar ----
        if self.master_calendar is not None:
            df = df.merge(self.master_calendar, on='Calendar_Date', how='left')
            df['Is_Holiday'] = df['Is_Holiday'].fillna(0)

            # Hard zero on observed federal holidays (Dec 25, Jan 1, etc.)
            forecast[df['Is_Holiday'].values == 1] = 0.0

            # ---- Holiday distortion ----
            holiday_dates_forecast = self.master_calendar.loc[
                (self.master_calendar['Is_Holiday'] == 1) &
                (self.master_calendar['Calendar_Date'] >= start_date) &
                (self.master_calendar['Calendar_Date'] <= future_dates[-1])
            ]['Calendar_Date'].tolist()

            lookback_start = start_date - pd.Timedelta(
                days=(self.holiday_window_before + 1) * 2
            )
            holiday_dates_nearby = self.master_calendar.loc[
                (self.master_calendar['Is_Holiday'] == 1) &
                (self.master_calendar['Calendar_Date'] >= lookback_start) &
                (self.master_calendar['Calendar_Date'] < start_date)
            ]['Calendar_Date'].tolist()

            all_relevant_holidays = holiday_dates_nearby + holiday_dates_forecast
            distortion            = np.ones(len(df))

            for hdate in all_relevant_holidays:
                holiday_dow = hdate.dayofweek
                if holiday_dow in [5, 6]:
                    continue

                for offset in range(
                    -self.holiday_window_before,
                     self.holiday_window_after + 1
                ):
                    if offset == 0:
                        continue

                    candidate  = hdate
                    steps      = 0
                    direction  = 1 if offset > 0 else -1
                    abs_offset = abs(offset)
                    while steps < abs_offset:
                        candidate = candidate + pd.Timedelta(days=direction)
                        if candidate.dayofweek not in [5, 6]:
                            steps += 1

                    row_idx = df.index[df['Calendar_Date'] == candidate].tolist()
                    if not row_idx:
                        continue
                    idx    = row_idx[0]
                    factor = self.distortion_table_.get(
                        (holiday_dow, offset), 1.0
                    )
                    distortion[idx] = factor

            forecast = forecast * distortion

        df['forecast'] = forecast
        return df

In [14]:
#///////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////
#                          StructuralLoanForecaster_Switch (V2 — Regime-Adaptive)
#
#   December → Model C
#   All other months → Model A with regime-adaptive parameters
#   Regime detected from pipeline leading indicators (1-month lead)
#///////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////

class StructuralLoanForecaster_Switch:
    """
    Switching forecaster:
        December  → Model C (December specialist)
        All else  → Model A with regime-adaptive parameters

    The regime detector uses pipeline leading indicators to automatically
    select the right Model A parameter set (Regime1 vs Regime2 vs January).
    """

    def __init__(self, today, target='total_loan_amount'):
        self.today            = pd.to_datetime(today)
        self.target           = target
        self.model_c          = None
        self.holiday_calendar = None
        self.regime_detector  = LeadingIndicatorRegimeDetector(threshold=0.30)
        self.fitted           = False

        # Store training data + pipeline data for re-fitting Model A per regime
        self._train_df        = None
        self._holiday_cal     = None
        self._pipeline_monthly = None

    @staticmethod
    def _count_biz_days(year, month, holiday_calendar):
        month_start = pd.Timestamp(year=year, month=month, day=1)
        month_end   = month_start + pd.offsets.MonthEnd(0)
        all_days    = pd.date_range(start=month_start, end=month_end, freq='D')
        holidays    = set(pd.to_datetime(
            holiday_calendar.loc[holiday_calendar['Is_Holiday'] == 1, 'Calendar_Date']
        ))
        return len([d for d in all_days if d.dayofweek not in [5, 6] and d not in holidays])

    def fit(self, df, holiday_calendar, pipeline_monthly_df):
        """
        Parameters
        ----------
        df : pd.DataFrame
            Historical loan data (Calendar_Date, total_loan_amount, etc.)
        holiday_calendar : pd.DataFrame
            Calendar_Date + Is_Holiday columns
        pipeline_monthly_df : pd.DataFrame
            Monthly aggregated pipeline indicators with growth rates
        """
        self.holiday_calendar = holiday_calendar.copy()
        self.holiday_calendar['Calendar_Date'] = pd.to_datetime(
            self.holiday_calendar['Calendar_Date']
        )

        self._train_df         = df.copy()
        self._holiday_cal      = holiday_calendar.copy()
        self._pipeline_monthly = pipeline_monthly_df.copy()

        # ── Pre-fit Model C (always the same params) ─────────────────────
        self.model_c = StructuralLoanForecaster_C()
        self.model_c.fit(df, holiday_calendar)

        self.fitted = True
        return self

    def _build_model_a(self, regime_params):
        """Fit a fresh Model A with the given regime parameters."""
        model = StructuralLoanForecaster_A(target=self.target, **regime_params)
        model.fit(self._train_df, self._holiday_cal)
        return model

    def forecast_from_date(self, start_date, months_forward=1):
        """
        For each month in the forecast window:
          - December → Model C
          - Otherwise → detect regime from pipeline data → fit Model A with regime params
        """
        if not self.fitted:
            raise Exception('Model must be fit before forecasting.')

        start_date = pd.to_datetime(start_date)
        records    = []

        for i in range(months_forward):
            if i == 0:
                period_start = start_date
            else:
                period_start = (start_date + pd.DateOffset(months=i)).replace(day=1)

            month_start = period_start.replace(day=1)
            period_end  = month_start + pd.offsets.MonthEnd(0)

            year  = month_start.year
            month = month_start.month
            n_biz = self._count_biz_days(year, month, self.holiday_calendar)

            if month == 12:
                # ── December → Model C ───────────────────────────────────
                print(f"  [Switch] {year}-{month:02d} → December → Model C selected")
                model = self.model_c
                model_choice = 'C'

            else:
                # ── Regime detection → Model A with adaptive params ──────
                regime, composite, regime_params = self.regime_detector.detect_regime(
                    forecast_month=month,
                    pipeline_monthly_df=self._pipeline_monthly
                )
                model = self._build_model_a(regime_params)
                model_choice = f'A({regime})'
                print(
                    f"  [Switch] {year}-{month:02d} → {n_biz} biz days → "
                    f"Model A [{regime}] selected"
                )

            pred = model.forecast_from_date(start_date=period_start, months_forward=1)
            pred = pred[
                (pred['Calendar_Date'] >= period_start) &
                (pred['Calendar_Date'] <= period_end)
            ].copy()

            pred['model_used'] = model_choice
            pred['n_biz_days'] = n_biz
            records.append(pred)

        return pd.concat(records, ignore_index=True)

In [15]:
#////////////////////////////////////////////////////////////////////////////////////////////////////////
#                                   MAIN EXECUTION — V2 (Regime-Adaptive)
#////////////////////////////////////////////////////////////////////////////////////////////////////////

# ── Business day helpers ──────────────────────────────────────────────────────
def add_business_days(start_date, n_days):
    current = pd.to_datetime(start_date)
    days_added = 0
    while days_added < n_days:
        current += pd.Timedelta(days=1)
        if current.dayofweek not in [5, 6]:
            days_added += 1
    return current

KNOWN_THROUGH  = add_business_days(TODAY, 3)
FORECAST_START = add_business_days(TODAY, 4)
OUTPUT_THROUGH = add_business_days(TODAY, 29)

print(f"TODAY          : {TODAY.date()}")
print(f"Actuals through: {KNOWN_THROUGH.date()}")
print(f"Forecast start : {FORECAST_START.date()}")
print(f"Output through : {OUTPUT_THROUGH.date()}")

# ── Build holiday calendar ────────────────────────────────────────────────────
holiday_calendar = build_us_holiday_calendar(start_date='2021-01-01', end_date='2030-12-31')

# ── Prepare pipeline monthly indicators ───────────────────────────────────────
pipeline_monthly = prepare_pipeline_monthly(pipeline_wholesale)

# Only use complete months (exclude current partial month)
last_complete_month = (TODAY.replace(day=1) - pd.Timedelta(days=1)).to_period('M')
pipeline_monthly_complete = pipeline_monthly[
    pipeline_monthly['year_month'] <= last_complete_month
].copy()

print(f"\nPipeline months available: {len(pipeline_monthly_complete)}")
print(f"Latest complete month: {pipeline_monthly_complete['year_month'].max()}")
print(f"Latest indicators:")
latest = pipeline_monthly_complete.iloc[-1]
print(f"  application_count_per_bizday:                        {latest['application_count_per_bizday']:.1f}")
print(f"  approval_events_per_bizday_growth_1m:                {latest['approval_events_per_bizday_growth_1m']:+.3f}")
print(f"  underwriting_submission_events_per_bizday_growth_1m: {latest['underwriting_submission_events_per_bizday_growth_1m']:+.3f}")

# ── Training data ─────────────────────────────────────────────────────────────
train_wholesale = wholesale_historical[
    (wholesale_historical['Calendar_Date'] >= '2021-09-01') &
    (wholesale_historical['Calendar_Date'] <= KNOWN_THROUGH)
].drop(columns=['Is_Holiday'], errors='ignore').copy()
train_wholesale['Calendar_Date'] = pd.to_datetime(train_wholesale['Calendar_Date'])

# ── Fit switch model (V2 — regime-adaptive) ───────────────────────────────────
model_switch = StructuralLoanForecaster_Switch(today=FORECAST_START)
model_switch.fit(train_wholesale, holiday_calendar, pipeline_monthly_complete)

print(f"\n{'='*70}")
print(f"  GENERATING FORECAST")
print(f"{'='*70}")

pred_switch = model_switch.forecast_from_date(
    start_date=FORECAST_START,
    months_forward=2
)
pred_switch = pred_switch[pred_switch['Calendar_Date'] <= OUTPUT_THROUGH].copy()

# ── Pull actuals for known window ─────────────────────────────────────────────
actuals_window = wholesale_historical[
    (wholesale_historical['Calendar_Date'] >= TODAY) &
    (wholesale_historical['Calendar_Date'] <= KNOWN_THROUGH)
][['Calendar_Date', 'total_loan_amount']].copy()
actuals_window['Calendar_Date'] = pd.to_datetime(actuals_window['Calendar_Date'])
actuals_window = actuals_window.rename(columns={'total_loan_amount': 'amount'})
actuals_window['source']     = 'actual'
actuals_window['model_used'] = '—'
actuals_window['forecast']   = None

# ── Build combined output ─────────────────────────────────────────────────────
forecast_out = pred_switch[['Calendar_Date', 'forecast', 'model_used']].copy()
forecast_out['amount'] = None
forecast_out['source'] = 'forecast'

wholesale_combined = pd.concat(
    [actuals_window, forecast_out], ignore_index=True
).sort_values('Calendar_Date').reset_index(drop=True)

wholesale_combined['channel'] = 'Wholesale'
wholesale_combined = wholesale_combined[
    ~wholesale_combined['Calendar_Date'].dt.dayofweek.isin([5, 6])
].reset_index(drop=True)

# ── Print output ──────────────────────────────────────────────────────────────
dow_names = {0: 'Mon', 1: 'Tue', 2: 'Wed', 3: 'Thu', 4: 'Fri'}

def fmt(val):
    if val is None or pd.isna(val):
        return '         —'
    return f'{float(val):>12,.0f}'

header = (
    f"{'Date':<15} {'DOW':<6} {'Source':<10} {'Model':<16} "
    f"{'Channel':<14} {'Amount':>12}"
)
print(f"\n{header}")
print("-" * 80)

for _, row in wholesale_combined.iterrows():
    dow_label = dow_names.get(row['Calendar_Date'].dayofweek, '—')
    amount    = row['amount'] if row['source'] == 'actual' else row['forecast']
    print(
        f"{str(row['Calendar_Date'].date()):<15} "
        f"{dow_label:<6} "
        f"{row['source']:<10} "
        f"{str(row['model_used']):<16} "
        f"{row['channel']:<14} "
        f"{fmt(amount)}"
    )

print("-" * 80)
actual_total   = actuals_window['amount'].dropna().astype(float).sum()
forecast_total = pred_switch['forecast'].dropna().astype(float).sum()
print(f"{'TOTAL (actuals + forecast)':<52} {fmt(actual_total + forecast_total)}")

TODAY          : 2026-04-01
Actuals through: 2026-04-06
Forecast start : 2026-04-07
Output through : 2026-05-12

Pipeline months available: 25
Latest complete month: 2026-03
Latest indicators:
  application_count_per_bizday:                        170.8
  approval_events_per_bizday_growth_1m:                -0.121
  underwriting_submission_events_per_bizday_growth_1m: -0.062
  [Model C] cliff_ratio (post/pre Dec 18): 0.917
  [Model C] Distortion keys learned: [(0, -2), (0, -1), (0, 1), (0, 2), (0, 3), (0, 4), (0, 6), (4, -2), (4, -1), (4, 1), (4, 2), (4, 3), (4, 4), (4, 6)]

  GENERATING FORECAST
  [Regime] month=4  composite=-0.589  → Regime1  (apps/biz=170.8, appr_g=-0.121, uw_g=-0.062)
  [Switch] 2026-04 → 22 biz days → Model A [Regime1] selected
  [Regime] month=5  composite=-0.589  → Regime1  (apps/biz=170.8, appr_g=-0.121, uw_g=-0.062)
  [Switch] 2026-05 → 20 biz days → Model A [Regime1] selected

Date            DOW    Source     Model            Channel              Amount
----

In [16]:
#////////////////////////////////////////////////////////////////////////////////////////////////////////
#                                   REBUILD EVAL_DF TO COVER FULL REQUESTED DATE RANGE
#////////////////////////////////////////////////////////////////////////////////////////////////////////

VIEW_FROM = pd.to_datetime(TODAY)
VIEW_TO   = pd.to_datetime('2026-04-30')

print(f"Requested range: {VIEW_FROM.date()} → {VIEW_TO.date()}")

# ── Pull actuals for the full requested window ────────────────────────────────
full_actuals = wholesale_historical[
    (wholesale_historical['Calendar_Date'] >= TODAY) &
    (wholesale_historical['Calendar_Date'] <= VIEW_TO)
][['Calendar_Date', 'total_loan_amount']].copy()

full_actuals['Calendar_Date'] = pd.to_datetime(full_actuals['Calendar_Date'])
full_actuals = full_actuals.rename(columns={'total_loan_amount': 'actual'})

# ── Re-fit switch model ───────────────────────────────────────────────────────
model_switch_eval = StructuralLoanForecaster_Switch(today=FORECAST_START)
model_switch_eval.fit(
    wholesale_historical[
        (wholesale_historical['Calendar_Date'] >= '2021-09-01') &
        (wholesale_historical['Calendar_Date'] <= KNOWN_THROUGH)
    ].drop(columns=['Is_Holiday'], errors='ignore').copy(),
    holiday_calendar,
    pipeline_monthly_complete   # ← THIS WAS MISSING
)

# months_forward=2 ensures we always cover VIEW_TO even for mid-month starts
months_needed = (
    (VIEW_TO.year - FORECAST_START.year) * 12 +
    (VIEW_TO.month - FORECAST_START.month) + 2
)

pred_switch_eval = model_switch_eval.forecast_from_date(
    start_date=FORECAST_START,
    months_forward=months_needed
)

# Cap to VIEW_TO
pred_switch_eval = pred_switch_eval[pred_switch_eval['Calendar_Date'] <= VIEW_TO].copy()

# ── Build eval_df ─────────────────────────────────────────────────────────────
eval_df = full_actuals.merge(
    pred_switch_eval[['Calendar_Date', 'forecast', 'model_used']],
    on='Calendar_Date', how='inner'
)

eval_df['channel'] = 'Wholesale'
eval_df['dow']     = eval_df['Calendar_Date'].dt.dayofweek

# Business days with actual > 0 only
eval_df = eval_df[
    (~eval_df['Calendar_Date'].dt.dayofweek.isin([5, 6])) &
    (eval_df['actual'] > 0)
].reset_index(drop=True)

# ── Known window: replace forecast with actuals ───────────────────────────────
known_mask = eval_df['Calendar_Date'] <= KNOWN_THROUGH
eval_df.loc[known_mask, 'forecast']   = eval_df.loc[known_mask, 'actual']
eval_df.loc[known_mask, 'model_used'] = 'actual'

eval_df['diff']    = eval_df['forecast'] - eval_df['actual']
eval_df['pct_err'] = (eval_df['diff'] / eval_df['actual']) * 100

eval_df['source'] = 'forecast'
eval_df.loc[known_mask, 'source'] = 'actual'

print(f"eval_df covers: {eval_df['Calendar_Date'].min().date()} → {eval_df['Calendar_Date'].max().date()}")
print(f"eval_df rows  : {len(eval_df)}")


#////////////////////////////////////////////////////////////////////////////////////////////////////////
#                                   PRINT OUTPUT TABLE
#////////////////////////////////////////////////////////////////////////////////////////////////////////

dow_names = {0: 'Mon', 1: 'Tue', 2: 'Wed', 3: 'Thu', 4: 'Fri'}

def fmt(val):
    if val is None or pd.isna(val):
        return '—'
    return f'{float(val):,.0f}'

def fmt_pct(val):
    if val is None or pd.isna(val):
        return '—'
    return f'{float(val):+.1f}%'

# ── Known actuals window ──────────────────────────────────────────────────────
known_actuals_print = wholesale_historical[
    (wholesale_historical['Calendar_Date'] >= TODAY) &
    (wholesale_historical['Calendar_Date'] <= KNOWN_THROUGH) &
    (~wholesale_historical['Calendar_Date'].dt.dayofweek.isin([5, 6])) &
    (wholesale_historical['total_loan_amount'] > 0)
][['Calendar_Date', 'total_loan_amount']].copy()

known_actuals_print['Calendar_Date'] = pd.to_datetime(known_actuals_print['Calendar_Date'])
known_actuals_print = known_actuals_print.rename(columns={'total_loan_amount': 'actual'})
known_actuals_print['source']    = 'actual'
known_actuals_print['channel']   = 'Wholesale'
known_actuals_print['forecast']  = known_actuals_print['actual']
known_actuals_print['model_used']= 'actual'
known_actuals_print['diff']      = 0.0
known_actuals_print['pct_err']   = 0.0

# ── Forecast rows ─────────────────────────────────────────────────────────────
forecast_rows = eval_df[eval_df['source'] == 'forecast'].copy()

# ── Combine ───────────────────────────────────────────────────────────────────
print_df = pd.concat(
    [known_actuals_print, forecast_rows], ignore_index=True
).sort_values(
    ['Calendar_Date', 'source'], ascending=[True, True]
).drop_duplicates(
    subset='Calendar_Date', keep='first'
).reset_index(drop=True)

# ── Apply date band filter ────────────────────────────────────────────────────
print_df_view = print_df[
    (print_df['Calendar_Date'] >= VIEW_FROM) &
    (print_df['Calendar_Date'] <= VIEW_TO)
].copy().reset_index(drop=True)

# ── Totals & MAE/MAPE ─────────────────────────────────────────────────────────
forecast_mask_view = print_df_view['source'] == 'forecast'

total_actual   = print_df_view.loc[forecast_mask_view, 'actual'].sum()
total_forecast = print_df_view.loc[forecast_mask_view, 'forecast'].sum()
total_diff     = total_forecast - total_actual
total_pct      = (total_diff / total_actual) * 100 if total_actual > 0 else float('nan')

mae  = print_df_view.loc[forecast_mask_view, 'diff'].abs().mean()    if forecast_mask_view.any() else float('nan')
mape = print_df_view.loc[forecast_mask_view, 'pct_err'].abs().mean() if forecast_mask_view.any() else float('nan')

# ── Build table columns ───────────────────────────────────────────────────────
col_date      = []
col_dow       = []
col_source    = []
col_model     = []
col_channel   = []
col_actual    = []
col_forecast  = []
col_diff      = []
col_pct       = []

for _, row in print_df_view.iterrows():
    col_date.append(str(row['Calendar_Date'].date()))
    col_dow.append(dow_names.get(row['Calendar_Date'].dayofweek, '—'))
    col_source.append(row['source'])
    col_model.append(str(row['model_used']))
    col_channel.append(row['channel'])
    col_actual.append(fmt(row['actual']))
    col_forecast.append(fmt(row['forecast']))
    col_diff.append(fmt(row['diff']))
    col_pct.append(fmt_pct(row['pct_err']))

# ── TOTAL row ─────────────────────────────────────────────────────────────────
col_date.append(f'TOTAL  ({VIEW_FROM.strftime("%b %d")} → {VIEW_TO.strftime("%b %d")})')
col_dow.append('')
col_source.append('forecast only')
col_model.append('')
col_channel.append('Wholesale')
col_actual.append(fmt(total_actual))
col_forecast.append(fmt(total_forecast))
col_diff.append(fmt(total_diff))
col_pct.append(fmt_pct(total_pct))

# ── MAE row ───────────────────────────────────────────────────────────────────
col_date.append('MAE')
for lst in [col_dow, col_source, col_model, col_channel, col_actual]:
    lst.append('')
col_forecast.append(fmt(mae))
col_diff.append('')
col_pct.append('')

# ── MAPE row ──────────────────────────────────────────────────────────────────
col_date.append('MAPE')
for lst in [col_dow, col_source, col_model, col_channel, col_actual]:
    lst.append('')
col_forecast.append(fmt_pct(mape))
col_diff.append('')
col_pct.append('')

# ── Row fill colours ──────────────────────────────────────────────────────────
n_known_view    = print_df_view[print_df_view['source'] == 'actual'].shape[0]
n_forecast_view = print_df_view[print_df_view['source'] == 'forecast'].shape[0]
n_summary       = 3

fill_colors = (
    ['#f0f0f0'] * n_known_view    +
    ['white']   * n_forecast_view +
    ['#fffbea'] * n_summary
)

# ── Plotly table ──────────────────────────────────────────────────────────────
table_fig = go.Figure(data=[go.Table(
    columnwidth = [110, 50, 80, 80, 80, 100, 100, 100, 80],
    header = dict(
        values = [
            '<b>Date</b>', '<b>DOW</b>', '<b>Source</b>', '<b>Model</b>',
            '<b>Channel</b>', '<b>Actual</b>',
            '<b>Forecast</b>', '<b>Diff</b>', '<b>Pct</b>'
        ],
        fill_color = '#2a3f5f',
        font       = dict(color='white', size=11),
        align      = ['left', 'center', 'center', 'center', 'center',
                      'right', 'right', 'right', 'right'],
        height     = 30
    ),
    cells = dict(
        values = [
            col_date, col_dow, col_source, col_model, col_channel,
            col_actual, col_forecast, col_diff, col_pct
        ],
        fill_color = [fill_colors] * 9,
        font       = dict(color='black', size=11),
        align      = ['left', 'center', 'center', 'center', 'center',
                      'right', 'right', 'right', 'right'],
        height     = 26
    )
)])

table_fig.update_layout(
    title = dict(
        text = (
            f'Wholesale — Actuals vs Forecast | '
            f'{VIEW_FROM.strftime("%b %d, %Y")} → {VIEW_TO.strftime("%b %d, %Y")}'
        ),
        font = dict(size=13)
    ),
    width  = 1000,
    height = 80 + (n_known_view + n_forecast_view + n_summary) * 28,
    margin = dict(l=10, r=10, t=50, b=10)
)

table_fig.show()


#////////////////////////////////////////////////////////////////////////////////////////////////////////
#                                   PLOT — ACTUALS vs FORECAST (SWITCH MODEL)
#////////////////////////////////////////////////////////////////////////////////////////////////////////

fig = go.Figure()

# ── Actuals line ──────────────────────────────────────────────────────────────
fig.add_trace(go.Scatter(
    x      = print_df['Calendar_Date'],
    y      = print_df['actual'],
    mode   = 'lines+markers',
    name   = 'Actual',
    line   = dict(color='black', width=2),
    marker = dict(size=6)
))

# ── Forecast line (colour-coded by model) ─────────────────────────────────────
forecast_only = print_df[print_df['source'] == 'forecast'].copy()

# Extract unique model labels from forecast data
model_colors = {
    'C': 'green',
}
# Default color for any A(Regime*) variant
for model_label in forecast_only['model_used'].unique():
    if 'Regime1' in str(model_label):
        model_colors[model_label] = 'steelblue'
    elif 'Regime2' in str(model_label):
        model_colors[model_label] = 'darkorange'
    elif 'January' in str(model_label):
        model_colors[model_label] = 'gold'
    elif model_label == 'C':
        model_colors[model_label] = 'green'
    else:
        model_colors[model_label] = 'grey'

for model_label in forecast_only['model_used'].unique():
    mask = forecast_only['model_used'] == model_label
    if mask.any():
        color = model_colors.get(model_label, 'grey')
        fig.add_trace(go.Scatter(
            x      = forecast_only.loc[mask, 'Calendar_Date'],
            y      = forecast_only.loc[mask, 'forecast'],
            mode   = 'lines+markers',
            name   = f'Forecast ({model_label})',
            line   = dict(color=color, width=2.5, dash='dash'),
            marker = dict(size=6)
        ))

fig.add_vline(
    x          = str(FORECAST_START.date()),
    line_width = 1.5,
    line_dash  = 'dash',
    line_color = 'grey'
)
fig.add_annotation(
    x=str(FORECAST_START.date()), y=1, yref='paper',
    text='Forecast Start', showarrow=False,
    xanchor='left', font=dict(color='grey', size=11)
)
fig.add_vrect(
    x0=str(TODAY.date()), x1=str(KNOWN_THROUGH.date()),
    fillcolor='lightgrey', opacity=0.2, layer='below', line_width=0
)
fig.add_annotation(
    x=str(TODAY.date()), y=1, yref='paper',
    text='Known Actuals', showarrow=False,
    xanchor='left', font=dict(color='grey', size=11)
)

fig.update_layout(
    title = dict(
        text=(
            f'Wholesale — Actuals vs Forecast (Regime-Adaptive)<br>'
            f'<sup>{TODAY.strftime("%b %d, %Y")} → {VIEW_TO.strftime("%b %d, %Y")}</sup>'
        ),
        font=dict(size=18)
    ),
    xaxis=dict(title='Date', tickformat='%b %d', tickangle=-45,
               showgrid=True, gridcolor='lightgrey'),
    yaxis=dict(title='Loan Volume ($)', tickformat='$,.0f',
               showgrid=True, gridcolor='lightgrey'),
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='left', x=0),
    hovermode='x unified', plot_bgcolor='white', width=1200, height=550
)

fig.show()

Requested range: 2026-04-01 → 2026-04-30


  [Model C] cliff_ratio (post/pre Dec 18): 0.917
  [Model C] Distortion keys learned: [(0, -2), (0, -1), (0, 1), (0, 2), (0, 3), (0, 4), (0, 6), (4, -2), (4, -1), (4, 1), (4, 2), (4, 3), (4, 4), (4, 6)]
  [Regime] month=4  composite=-0.589  → Regime1  (apps/biz=170.8, appr_g=-0.121, uw_g=-0.062)
  [Switch] 2026-04 → 22 biz days → Model A [Regime1] selected
  [Regime] month=5  composite=-0.589  → Regime1  (apps/biz=170.8, appr_g=-0.121, uw_g=-0.062)
  [Switch] 2026-05 → 20 biz days → Model A [Regime1] selected
eval_df covers: 2026-04-07 → 2026-04-27
eval_df rows  : 15


In [17]:
#////////////////////////////////////////////////////////////////////////////////////////////////////////
#                                   OUTPUT: ACTUALS (known) + FORECAST (rest of month)
#////////////////////////////////////////////////////////////////////////////////////////////////////////

VIEW_FROM = pd.to_datetime(TODAY)
VIEW_TO   = pd.to_datetime(TODAY) + pd.offsets.MonthEnd(0)  # end of current month

print(f"Requested range: {VIEW_FROM.date()} → {VIEW_TO.date()}")

# ── Re-fit switch model ───────────────────────────────────────────────────────
model_switch_eval = StructuralLoanForecaster_Switch(today=FORECAST_START)
model_switch_eval.fit(
    wholesale_historical[
        (wholesale_historical['Calendar_Date'] >= '2021-09-01') &
        (wholesale_historical['Calendar_Date'] <= KNOWN_THROUGH)
    ].drop(columns=['Is_Holiday'], errors='ignore').copy(),
    holiday_calendar,
    pipeline_monthly_complete
)

months_needed = (
    (VIEW_TO.year - FORECAST_START.year) * 12 +
    (VIEW_TO.month - FORECAST_START.month) + 2
)

pred_switch_eval = model_switch_eval.forecast_from_date(
    start_date=FORECAST_START,
    months_forward=months_needed
)
pred_switch_eval = pred_switch_eval[pred_switch_eval['Calendar_Date'] <= VIEW_TO].copy()

# ── Known actuals (TODAY → KNOWN_THROUGH) ─────────────────────────────────────
known_actuals_print = wholesale_historical[
    (wholesale_historical['Calendar_Date'] >= TODAY) &
    (wholesale_historical['Calendar_Date'] <= KNOWN_THROUGH) &
    (~wholesale_historical['Calendar_Date'].dt.dayofweek.isin([5, 6])) &
    (wholesale_historical['total_loan_amount'] > 0)
][['Calendar_Date', 'total_loan_amount']].copy()

known_actuals_print['Calendar_Date'] = pd.to_datetime(known_actuals_print['Calendar_Date'])
known_actuals_print = known_actuals_print.rename(columns={'total_loan_amount': 'amount'})
known_actuals_print['source']     = 'actual'
known_actuals_print['channel']    = 'Wholesale'
known_actuals_print['model_used'] = 'actual'

# ── Forecast rows (FORECAST_START → VIEW_TO) ─────────────────────────────────
forecast_rows = pred_switch_eval[['Calendar_Date', 'forecast', 'model_used']].copy()
forecast_rows = forecast_rows[
    (~forecast_rows['Calendar_Date'].dt.dayofweek.isin([5, 6])) &
    (forecast_rows['forecast'] > 0)
].reset_index(drop=True)
forecast_rows['amount']  = forecast_rows['forecast']
forecast_rows['source']  = 'forecast'
forecast_rows['channel'] = 'Wholesale'

# ── Combine ───────────────────────────────────────────────────────────────────
print_df = pd.concat(
    [known_actuals_print[['Calendar_Date', 'amount', 'source', 'model_used', 'channel']],
     forecast_rows[['Calendar_Date', 'amount', 'source', 'model_used', 'channel']]],
    ignore_index=True
).sort_values('Calendar_Date').drop_duplicates(subset='Calendar_Date', keep='first').reset_index(drop=True)

print_df_view = print_df[
    (print_df['Calendar_Date'] >= VIEW_FROM) &
    (print_df['Calendar_Date'] <= VIEW_TO)
].copy().reset_index(drop=True)

# ── Totals ────────────────────────────────────────────────────────────────────
actual_total   = print_df_view.loc[print_df_view['source'] == 'actual',   'amount'].sum()
forecast_total = print_df_view.loc[print_df_view['source'] == 'forecast', 'amount'].sum()
grand_total    = actual_total + forecast_total

print(f"print_df covers: {print_df_view['Calendar_Date'].min().date()} → {print_df_view['Calendar_Date'].max().date()}")
print(f"print_df rows  : {len(print_df_view)}")

# ── Build table columns ───────────────────────────────────────────────────────
dow_names = {0: 'Mon', 1: 'Tue', 2: 'Wed', 3: 'Thu', 4: 'Fri'}

def fmt(val):
    if val is None or pd.isna(val):
        return '—'
    return f'{float(val):,.0f}'

col_date, col_dow, col_source, col_model, col_channel, col_amount = [], [], [], [], [], []

for _, row in print_df_view.iterrows():
    col_date.append(str(row['Calendar_Date'].date()))
    col_dow.append(dow_names.get(row['Calendar_Date'].dayofweek, '—'))
    col_source.append(row['source'])
    col_model.append(str(row['model_used']))
    col_channel.append(row['channel'])
    col_amount.append(fmt(row['amount']))

# ── TOTAL row ─────────────────────────────────────────────────────────────────
col_date.append(f'TOTAL  ({VIEW_FROM.strftime("%b %d")} → {VIEW_TO.strftime("%b %d")})')
col_dow.append('')
col_source.append('actuals + forecast')
col_model.append('')
col_channel.append('Wholesale')
col_amount.append(fmt(grand_total))

# ── Row fill colours ──────────────────────────────────────────────────────────
n_known_view    = print_df_view[print_df_view['source'] == 'actual'].shape[0]
n_forecast_view = print_df_view[print_df_view['source'] == 'forecast'].shape[0]
n_summary       = 1

fill_colors = (
    ['#f0f0f0'] * n_known_view    +
    ['white']   * n_forecast_view +
    ['#fffbea'] * n_summary
)

# ── Plotly table ──────────────────────────────────────────────────────────────
table_fig = go.Figure(data=[go.Table(
    columnwidth = [110, 50, 80, 80, 80, 100],
    header = dict(
        values = ['<b>Date</b>', '<b>DOW</b>', '<b>Source</b>', '<b>Model</b>',
                  '<b>Channel</b>', '<b>Amount</b>'],
        fill_color = '#2a3f5f',
        font       = dict(color='white', size=11),
        align      = ['left', 'center', 'center', 'center', 'center', 'right'],
        height     = 30
    ),
    cells = dict(
        values = [col_date, col_dow, col_source, col_model, col_channel, col_amount],
        fill_color = [fill_colors] * 6,
        font       = dict(color='black', size=11),
        align      = ['left', 'center', 'center', 'center', 'center', 'right'],
        height     = 26
    )
)])

table_fig.update_layout(
    title = dict(
        text = (f'Wholesale — Actuals + Forecast | '
                f'{VIEW_FROM.strftime("%b %d, %Y")} → {VIEW_TO.strftime("%b %d, %Y")}'),
        font = dict(size=13)
    ),
    width  = 900,
    height = 80 + (n_known_view + n_forecast_view + n_summary) * 28,
    margin = dict(l=10, r=10, t=50, b=10)
)
table_fig.show()

# ── Line chart ────────────────────────────────────────────────────────────────
fig = go.Figure()

# Actuals
actual_df = print_df_view[print_df_view['source'] == 'actual']
fig.add_trace(go.Scatter(
    x=actual_df['Calendar_Date'], y=actual_df['amount'],
    mode='lines+markers', name='Actual',
    line=dict(color='black', width=2), marker=dict(size=6)
))

# Forecast by model
fc_df = print_df_view[print_df_view['source'] == 'forecast']
model_colors = {}
for ml in fc_df['model_used'].unique():
    if 'Regime1' in str(ml):   model_colors[ml] = 'steelblue'
    elif 'Regime2' in str(ml): model_colors[ml] = 'darkorange'
    elif 'January' in str(ml): model_colors[ml] = 'gold'
    elif ml == 'C':            model_colors[ml] = 'green'
    else:                      model_colors[ml] = 'grey'

for ml in fc_df['model_used'].unique():
    mask = fc_df['model_used'] == ml
    fig.add_trace(go.Scatter(
        x=fc_df.loc[mask, 'Calendar_Date'], y=fc_df.loc[mask, 'amount'],
        mode='lines+markers', name=f'Forecast ({ml})',
        line=dict(color=model_colors.get(ml, 'grey'), width=2.5, dash='dash'),
        marker=dict(size=6)
    ))

fig.add_vline(x=str(FORECAST_START.date()), line_width=1.5, line_dash='dash', line_color='grey')
fig.add_annotation(x=str(FORECAST_START.date()), y=1, yref='paper',
                   text='Forecast Start', showarrow=False, xanchor='left',
                   font=dict(color='grey', size=11))
fig.add_vrect(x0=str(TODAY.date()), x1=str(KNOWN_THROUGH.date()),
              fillcolor='lightgrey', opacity=0.2, layer='below', line_width=0)
fig.add_annotation(x=str(TODAY.date()), y=1, yref='paper',
                   text='Known Actuals', showarrow=False, xanchor='left',
                   font=dict(color='grey', size=11))

fig.update_layout(
    title=dict(text=(f'Wholesale — Actuals + Forecast (Regime-Adaptive)<br>'
                     f'<sup>{TODAY.strftime("%b %d, %Y")} → {VIEW_TO.strftime("%b %d, %Y")}</sup>'),
               font=dict(size=18)),
    xaxis=dict(title='Date', tickformat='%b %d', tickangle=-45, showgrid=True, gridcolor='lightgrey'),
    yaxis=dict(title='Loan Volume ($)', tickformat='$,.0f', showgrid=True, gridcolor='lightgrey'),
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='left', x=0),
    hovermode='x unified', plot_bgcolor='white', width=1200, height=550
)
fig.show()

Requested range: 2026-04-01 → 2026-04-30
  [Model C] cliff_ratio (post/pre Dec 18): 0.917
  [Model C] Distortion keys learned: [(0, -2), (0, -1), (0, 1), (0, 2), (0, 3), (0, 4), (0, 6), (4, -2), (4, -1), (4, 1), (4, 2), (4, 3), (4, 4), (4, 6)]
  [Regime] month=4  composite=-0.589  → Regime1  (apps/biz=170.8, appr_g=-0.121, uw_g=-0.062)
  [Switch] 2026-04 → 22 biz days → Model A [Regime1] selected
  [Regime] month=5  composite=-0.589  → Regime1  (apps/biz=170.8, appr_g=-0.121, uw_g=-0.062)
  [Switch] 2026-05 → 20 biz days → Model A [Regime1] selected
print_df covers: 2026-04-01 → 2026-04-30
print_df rows  : 22
